<a href="https://colab.research.google.com/github/Innovatewithapple/bert-dense-retriever-benchmark/blob/main/EvaluateBeirBenchMark_TrecCovidDataset_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch datasets beir

In [ ]:
pip install rank-bm25

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModel
from datasets import load_dataset
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

In [ ]:
!wget https://huggingface.co/Innovatewithapple/bert-dense-retriever/resolve/main/configuration_bert_retriever.py
!wget https://huggingface.co/Innovatewithapple/bert-dense-retriever/resolve/main/modeling_bert_retriever.py

In [ ]:
print("--- Loading Custom Model & Tokenizer ---")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import torch
from transformers import AutoTokenizer
from datasets import load_dataset
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from sentence_transformers import CrossEncoder

# --- 1. Import Your Local Classes ---
from configuration_bert_retriever import BertRetrieverConfig
from modeling_bert_retriever import BertRetriever

device = "cuda" if torch.cuda.is_available() else "cpu"

print("--- 1. Initializing Local Architecture & Checkpoint ---")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config = BertRetrieverConfig(
    backbone_name="bert-base-uncased",
    hidden_size=768,
    dropout=0.2,
)

model = BertRetriever(config).to(device)
checkpoint = torch.load(
    "/content/best_model_epochj6_9693.pth",
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# --- 2. Define the Custom Wrapper Class ---
# CHANGE 1: BeIR expects 'encode_passages' as the method name, not 'encode_corpus'
class CustomHFModelWrapper:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    @torch.no_grad()
    def encode_queries(self, queries, batch_size=16, **kwargs):
        embeddings = []
        for i in range(0, len(queries), batch_size):
            batch = queries[i:i+batch_size]
            inputs = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(self.device)
            out = self.model(**inputs)
            embeddings.append(out.cpu())
        return torch.cat(embeddings, dim=0).numpy()

    @torch.no_grad()
    def encode_corpus(self, passages, batch_size=16, **kwargs):
        text_passages = [doc["text"] for doc in passages]
        embeddings = []
        for i in range(0, len(text_passages), batch_size):
            batch = text_passages[i:i+batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(self.device)
            out = self.model(**inputs)
            embeddings.append(out.cpu())
        return torch.cat(embeddings, dim=0).numpy()

# --- 3. Stream Dataset (TREC-COVID Medical - FULL EVALUATION) ---
print("\n--- 2. Streaming FULL TREC-COVID Dataset from Hugging Face ---")
hf_corpus = load_dataset("BeIR/trec-covid", "corpus", split="corpus")
hf_queries = load_dataset("BeIR/trec-covid", "queries", split="queries")
hf_qrels = load_dataset("BeIR/trec-covid-qrels", split="test")

# No slicing limit used here - gathers every single query natively
all_query_ids = set([str(row["query-id"]) for row in hf_qrels])

queries = {str(row["_id"]): row["text"] for row in hf_queries if str(row["_id"]) in all_query_ids}

qrels = {}
needed_corpus_ids = set()
for row in hf_qrels:
    q_id = str(row["query-id"])
    doc_id = str(row["corpus-id"])
    if q_id in all_query_ids:
        if q_id not in qrels:
            qrels[q_id] = {}
        qrels[q_id][doc_id] = int(row["score"])
        needed_corpus_ids.add(doc_id)

corpus = {str(row["_id"]): {"text": row["text"], "title": row["title"]} for row in hf_corpus if str(row["_id"]) in needed_corpus_ids}

print(f"Loaded the ENTIRE TREC-COVID dataset: {len(queries)} queries and {len(corpus)} passages.")


# --- 4. Run Retrieval and Rerank ---
wrapped_model = DRES(CustomHFModelWrapper(model, tokenizer, device), batch_size=16)

print("\n--- Phase 1: Fast Vector Retrieval (Top 100) ---")
initial_results = wrapped_model.search(
    corpus,
    queries,
    top_k=100,
    score_function="cos_sim"
)

print("\n--- Phase 2: Cross-Encoder Re-ranking ---")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

reranked_results = {}
for query_id, doc_scores in initial_results.items():
    query_text = queries[query_id]
    pairs = [[query_text, corpus[doc_id]["text"]] for doc_id in doc_scores.keys()]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=True)
    reranked_results[query_id] = {doc_id: float(score) for doc_id, score in zip(doc_scores.keys(), scores)}

print("\n--- Phase 3: Calculating Final Metrics ---")
# Using "cos_sim" to maintain metrics consistency with Phase 1 configurations
retriever = EvaluateRetrieval(wrapped_model, score_function="cos_sim")
ndcg, _map, recall, precision = retriever.evaluate(qrels, reranked_results, retriever.k_values)

print("\n=============================================")
print("         RE-RANKED PIPELINE RESULTS          ")
print("=============================================")
print(f"NDCG@10:  {ndcg['NDCG@10']:.4f}")
print(f"Recall@10: {recall['Recall@10']:.4f}")
print("=============================================")


--- 1. Initializing Local Architecture & Checkpoint ---

--- 2. Streaming FULL TREC-COVID Dataset from Hugging Face ---
Loaded the ENTIRE TREC-COVID dataset: 50 queries and 35480 passages.

--- Phase 1: Fast Vector Retrieval (Top 100) ---

--- Phase 2: Cross-Encoder Re-ranking ---


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


--- Phase 3: Calculating Final Metrics ---

         RE-RANKED PIPELINE RESULTS          
NDCG@10:  0.6483
Recall@10: 0.0174


In [ ]:
import torch
import numpy as np
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer
from datasets import load_dataset
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from sentence_transformers import CrossEncoder

# --- 1. Import Your Local Classes ---
from configuration_bert_retriever import BertRetrieverConfig
from modeling_bert_retriever import BertRetriever

device = "cuda" if torch.cuda.is_available() else "cpu"

print("--- 1. Initializing Local Architecture & Checkpoint ---")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config = BertRetrieverConfig(
    backbone_name="bert-base-uncased",
    hidden_size=768,
    dropout=0.2,
)

model = BertRetriever(config).to(device)
checkpoint = torch.load(
    "/content/best_model_epochj6_9693.pth",
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# --- 2. Define the Custom Wrapper Class ---
# CHANGE 1: BeIR expects 'encode_passages' as the method name, not 'encode_corpus'
class CustomHFModelWrapper:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    @torch.no_grad()
    def encode_queries(self, queries, batch_size=16, **kwargs):
        embeddings = []
        for i in range(0, len(queries), batch_size):
            batch = queries[i:i+batch_size]
            inputs = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(self.device)
            out = self.model(**inputs)
            embeddings.append(out.cpu())
        return torch.cat(embeddings, dim=0).numpy()

    @torch.no_grad()
    def encode_corpus(self, passages, batch_size=16, **kwargs):
        text_passages = [doc["text"] for doc in passages]
        embeddings = []
        for i in range(0, len(text_passages), batch_size):
            batch = text_passages[i:i+batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(self.device)
            out = self.model(**inputs)
            embeddings.append(out.cpu())
        return torch.cat(embeddings, dim=0).numpy()

# --- 3. Stream Dataset (TREC-COVID Medical - FULL EVALUATION) ---
print("\n--- 2. Streaming FULL TREC-COVID Dataset from Hugging Face ---")
hf_corpus = load_dataset("BeIR/trec-covid", "corpus", split="corpus")
hf_queries = load_dataset("BeIR/trec-covid", "queries", split="queries")
hf_qrels = load_dataset("BeIR/trec-covid-qrels", split="test")

# No slicing limit used here - gathers every single query natively
all_query_ids = set([str(row["query-id"]) for row in hf_qrels])

queries = {str(row["_id"]): row["text"] for row in hf_queries if str(row["_id"]) in all_query_ids}

qrels = {}
needed_corpus_ids = set()
for row in hf_qrels:
    q_id = str(row["query-id"])
    doc_id = str(row["corpus-id"])
    if q_id in all_query_ids:
        if q_id not in qrels:
            qrels[q_id] = {}
        qrels[q_id][doc_id] = int(row["score"])
        needed_corpus_ids.add(doc_id)

corpus = {str(row["_id"]): {"text": row["text"], "title": row["title"]} for row in hf_corpus if str(row["_id"]) in needed_corpus_ids}

print(f"Loaded the ENTIRE TREC-COVID dataset: {len(queries)} queries and {len(corpus)} passages.")

# --- 4. Hybrid Retrieval and Reranking Pipeline ---
print("\n--- Phase 1A: Running Vector Search (Dense) ---")
wrapped_model = DRES(CustomHFModelWrapper(model, tokenizer, device), batch_size=16)
# Get top 100 semantic candidates from your BERT model
dense_results = wrapped_model.search(corpus, queries, top_k=100, score_function="cos_sim")

print("\n--- Phase 1B: Building Local BM25 Index (Sparse) ---")
# FIXED: Merging 'title' and 'text' fields ensures BM25 actually indexes the medical terms
corpus_ids = list(corpus.keys())
tokenized_corpus = [
    f"{corpus[doc_id]['title']} {corpus[doc_id]['text']}".lower().split(" ")
    for doc_id in corpus_ids
]
bm25 = BM25Okapi(tokenized_corpus)

print("--- Phase 1C: Running Keyword Search (BM25) ---")
sparse_results = {}
for query_id, query_text in queries.items():
    tokenized_query = query_text.lower().split(" ")
    # Get scores for all documents in the corpus based on keyword hits
    doc_scores = bm25.get_scores(tokenized_query)

    # Sort and capture the top 100 document IDs matching this keyword query
    top_indices = np.argsort(doc_scores)[::-1][:100]
    sparse_results[query_id] = {corpus_ids[idx]: float(doc_scores[idx]) for idx in top_indices}

print("\n--- Phase 2: Merging Candidates & Cross-Encoder Reranking ---")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
hybrid_reranked_results = {}

for query_id in queries.keys():
    query_text = queries[query_id]

    # Gather candidate pools from both retrievers
    dense_candidates = list(dense_results.get(query_id, {}).keys())
    sparse_candidates = list(sparse_results.get(query_id, {}).keys())

    # Merge candidates into a unique set to completely eliminate duplicates
    combined_candidates = list(set(dense_candidates + sparse_candidates))

    # Send the combined candidates to the Cross-Encoder for sorting
    pairs = [[query_text, corpus[doc_id]["text"]] for doc_id in combined_candidates]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=True)

    # Map the final scores back to their document IDs
    hybrid_reranked_results[query_id] = {doc_id: float(score) for doc_id, score in zip(combined_candidates, scores)}

print("\n--- Phase 3: Calculating Final Hybrid Pipeline Metrics ---")
retriever = EvaluateRetrieval(wrapped_model, score_function="cos_sim")
ndcg, _map, recall, precision = retriever.evaluate(qrels, hybrid_reranked_results, retriever.k_values)

print("\n=============================================")
print("         HYBRID PIPELINE RESULTS            ")
print("=============================================")
print(f"NDCG@10:    {ndcg['NDCG@10']:.4f}")
print(f"Recall@10:  {recall['Recall@10']:.4f}  (Mathematically capped low due to dataset size)")
print(f"Recall@100: {recall['Recall@100']:.4f} <- WATCH THIS NUMBER JUMP!")
print("=============================================")



--- 1. Initializing Local Architecture & Checkpoint ---

--- 2. Streaming FULL TREC-COVID Dataset from Hugging Face ---
Loaded the ENTIRE TREC-COVID dataset: 50 queries and 35480 passages.

--- Phase 1A: Running Vector Search (Dense) ---

--- Phase 1B: Building Local BM25 Index (Sparse) ---
--- Phase 1C: Running Keyword Search (BM25) ---

--- Phase 2: Merging Candidates & Cross-Encoder Reranking ---


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]


--- Phase 3: Calculating Final Hybrid Pipeline Metrics ---

         HYBRID PIPELINE RESULTS            
NDCG@10:    0.6868
Recall@10:  0.0181  (Mathematically capped low due to dataset size)
Recall@100: 0.1110 <- WATCH THIS NUMBER JUMP!
